# Modelo em ONNX para rodar no navegador

Converte o ConvNeXt-Tiny ajustado para ONNX com o exportador do próprio PyTorch, quantiza as camadas lineares em int8 e sobe os dois arquivos para a pasta `onnx/` do repositório do modelo. É o formato que o Transformers.js lê, então a demo roda no navegador de quem visita, sem servidor.

As camadas lineares concentram quase todos os pesos do ConvNeXt. As convoluções ficam em fp32, porque o operador de convolução quantizada não roda no WebAssembly do navegador.

O upload só acontece se as duas versões em ONNX derem o mesmo primeiro palpite do PyTorch em pelo menos 99% das fotos de teste. Roda em CPU, em uns 10 minutos.

In [ ]:
!pip install -q onnx onnxruntime onnxscript

O modelo ainda é privado e o upload escreve no repositório, então o login pede um token de escrita do Hugging Face.

In [ ]:
from huggingface_hub import notebook_login

notebook_login()

## Exportação

O modelo do `transformers` devolve um objeto com vários campos. O invólucro deixa só os logits, com os nomes de entrada e saída que o Transformers.js espera (`pixel_values` e `logits`). O tamanho do lote fica livre.

In [ ]:
from pathlib import Path

import onnx
import torch
from transformers import AutoModelForImageClassification

REPO = "juan1211/pratos-convnext-tiny-food101"
SAIDA = Path("onnx")
AMOSTRA = 500
MINIMO = 0.99


class Logits(torch.nn.Module):
    def __init__(self, modelo):
        super().__init__()
        self.modelo = modelo

    def forward(self, pixel_values):
        return self.modelo(pixel_values=pixel_values).logits


original = AutoModelForImageClassification.from_pretrained(REPO).eval()
SAIDA.mkdir(exist_ok=True)
torch.onnx.export(
    Logits(original),
    (torch.randn(2, 3, 224, 224),),
    SAIDA / "model.onnx",
    input_names=["pixel_values"],
    output_names=["logits"],
    dynamic_shapes={"pixel_values": {0: torch.export.Dim("lote", min=1, max=64)}},
    external_data=False,
)
onnx.checker.check_model(SAIDA / "model.onnx")
print("exportado:", round((SAIDA / "model.onnx").stat().st_size / 2**20, 1), "MB")

## Quantização

In [ ]:
from onnxruntime.quantization import QuantType, quantize_dynamic

quantize_dynamic(
    SAIDA / "model.onnx",
    SAIDA / "model_quantized.onnx",
    op_types_to_quantize=["MatMul", "Gemm"],
    weight_type=QuantType.QUInt8,
)
print("quantizado:", round((SAIDA / "model_quantized.onnx").stat().st_size / 2**20, 1), "MB")

## Conferência

As duas versões em ONNX rodam nas primeiras fotos do teste do Food-101 e são comparadas com o PyTorch.

In [ ]:
import numpy as np
import onnxruntime as ort
from datasets import load_dataset
from transformers import AutoImageProcessor

processador = AutoImageProcessor.from_pretrained(REPO)
fotos = load_dataset("ethz/food101", split=f"validation[:{AMOSTRA}]")
entradas = processador([f.convert("RGB") for f in fotos["image"]], return_tensors="np")["pixel_values"]
rotulos = np.array(fotos["label"])

with torch.inference_mode():
    palpite_original = original(pixel_values=torch.from_numpy(entradas)).logits.argmax(1).numpy()

concordancia = {}
for nome in ("model.onnx", "model_quantized.onnx"):
    sessao = ort.InferenceSession(SAIDA / nome, providers=["CPUExecutionProvider"])
    palpite = np.concatenate(
        [sessao.run(None, {"pixel_values": entradas[i : i + 50]})[0].argmax(1) for i in range(0, AMOSTRA, 50)]
    )
    concordancia[nome] = (palpite == palpite_original).mean()
    print(f"{nome}: mesmo palpite do PyTorch em {concordancia[nome]:.1%}, top-1 {(palpite == rotulos).mean():.1%}")
print(f"PyTorch: top-1 {(palpite_original == rotulos).mean():.1%}")

## Subir para o Hugging Face

In [ ]:
from huggingface_hub import upload_folder

assert min(concordancia.values()) >= MINIMO, f"conversão diverge do PyTorch: {concordancia}"
upload_folder(
    repo_id=REPO,
    folder_path=SAIDA,
    path_in_repo="onnx",
    allow_patterns=["model.onnx", "model_quantized.onnx"],
    commit_message="pesos em ONNX para o Transformers.js",
)